# 01_Ingestion_and_Parquet_Conversion

Select BDA Local (Python 3.11), or the project .venv Python environment, in VS Code. Run 01, 02, and 03 sequentially. All CSV inputs, Parquet outputs, reports, and staging files use the local project root (or BDA_PROJECT_DIR). No Colab connection, uploads, exports, or Drive mounting are used. The local prototype uses four Spark threads. Missing label dates are excluded; delay defaults to timestamp.


In [3]:
import os
import sys
import json
import subprocess
import tempfile
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

if os.name != "nt":
    raise RuntimeError("Select the local Windows BDA Local kernel in VS Code. This notebook does not use Colab.")
# Keep all reads/writes local to this checkout; override on another device if needed.
project_hint = os.environ.get("BDA_PROJECT_DIR")
project_cwd = Path.cwd().resolve()
base_dir = (Path(project_hint).expanduser().resolve() if project_hint else
            project_cwd.parent if project_cwd.name == "notebooks" else project_cwd)
if not (base_dir / "notebooks").is_dir():
    raise FileNotFoundError("Open the project root/notebooks directory or set BDA_PROJECT_DIR to the local checkout.")
raw_dir = base_dir / "datasets_raw"
parquet_dir = base_dir / "datasets_parquet"
output_dir = base_dir / "outputs_ml_graph"
runtime_dir = base_dir / ".runtime"
temp_dir = runtime_dir / "tmp"
temp_dir.mkdir(parents=True, exist_ok=True)
for name in ["TMP", "TEMP", "TMPDIR"]:
    os.environ[name] = str(temp_dir)
tempfile.tempdir = str(temp_dir)
java_candidates = sorted(p for p in runtime_dir.glob("jdk-*") if (p / "bin/java.exe").exists())
if not java_candidates:
    raise FileNotFoundError("The project-local Java runtime is missing from .runtime/jdk-*. Complete local setup first.")
os.environ["JAVA_HOME"] = str(java_candidates[-1])
os.environ["HADOOP_HOME"] = str(runtime_dir / "hadoop")
os.environ["SPARK_HOME"] = str(runtime_dir / "spark-home")
os.environ["PATH"] = (str(java_candidates[-1] / "bin") + os.pathsep +
                      str(runtime_dir / "hadoop/bin") + os.pathsep + os.environ["PATH"])
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
required_csvs = [
    "olist_customers_dataset.csv", "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv", "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv", "olist_orders_dataset.csv",
    "olist_products_dataset.csv", "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
]
missing = [name for name in required_csvs if not (raw_dir / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing local CSVs in {raw_dir}: {missing}")
output_dir.mkdir(parents=True, exist_ok=True)
DELAY_DEFINITION = os.environ.get("BDA_DELAY_DEFINITION", "timestamp")
if DELAY_DEFINITION not in {"timestamp", "calendar_date"}:
    raise ValueError("BDA_DELAY_DEFINITION must be timestamp or calendar_date")

def ensure_package(name, required_version=None):
    try:
        installed = version(name)
    except PackageNotFoundError:
        installed = None
    if installed is None or (required_version and installed != required_version):
        spec = f"{name}=={required_version}" if required_version else name
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", spec])

print(f"BDA LOCAL WINDOWS | Python: {sys.executable}")
print(f"All project data and outputs: {base_dir}")
print(f"Delay definition: {DELAY_DEFINITION}")
ensure_package("pyspark", "4.0.4")
ensure_package("pandas")
import pandas as pd
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = (SparkSession.builder.master("local[4]")
         .appName("Olist_Assignment_Pipeline")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.local.dir", str(runtime_dir / "spark-local"))
         .config("spark.sql.warehouse.dir", (runtime_dir / "spark-warehouse").as_uri())
         .config("spark.driver.bindAddress", "127.0.0.1")
         .config("spark.driver.host", "127.0.0.1")
         .config("spark.sql.session.timeZone", "America/Sao_Paulo")
         .getOrCreate())
print(f"Spark {spark.version}; master={spark.sparkContext.master} (single-machine prototype)")

def small_to_pandas(df):
    # Only aggregated reports are collected to the driver.
    return pd.DataFrame([row.asDict() for row in df.collect()], columns=df.columns)

def assert_unique(df, keys, name):
    if df.filter(F.expr(" OR ".join(f"`{key}` IS NULL" for key in keys))).limit(1).count():
        raise ValueError(f"{name}: null key in {keys}")
    if df.groupBy(*keys).count().filter(F.col("count") > 1).limit(1).count():
        raise ValueError(f"{name}: duplicate key {keys}")

def write_feature_parquet(df):
    import shutil
    import tempfile
    destination = output_dir / "ml_delivery_delay_dataset.parquet"
    expected_rows = df.count()
    with tempfile.TemporaryDirectory(prefix="olist_features_") as work:
        staged = Path(work) / destination.name
        df.write.mode("overwrite").option("compression", "snappy").parquet(str(staged))
        if spark.read.parquet(str(staged)).count() != expected_rows:
            raise ValueError("Staged feature Parquet row count mismatch")
        if destination.resolve().parent != output_dir.resolve():
            raise ValueError("Feature output path escaped output_dir")
        if destination.is_dir():
            shutil.rmtree(destination)
        elif destination.exists():
            destination.unlink()
        shutil.copytree(staged, destination)
        parts = list(destination.glob("*.parquet"))
        if not parts or any(p.stat().st_size == 0 for p in parts):
            raise ValueError("Feature Parquet has missing/empty parts")
        if spark.read.parquet(str(destination)).count() != expected_rows:
            raise ValueError("Copied feature Parquet row count mismatch")


BDA LOCAL WINDOWS | Python: d:\Private Project\bda\BDA_Olist_Project\.venv\Scripts\python.exe
All project data and outputs: D:\Private Project\bda\BDA_Olist_Project
Delay definition: timestamp
Spark 4.0.4; master=local[4] (single-machine prototype)


In [4]:
import csv
import re
import time
import shutil
import tempfile
from pyspark.sql.types import StructType, StructField, StringType

required_csvs = [
    "olist_customers_dataset.csv", "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv", "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv", "olist_orders_dataset.csv",
    "olist_products_dataset.csv", "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
]
missing = [name for name in required_csvs if not (raw_dir / name).is_file()]
if missing:
    raise FileNotFoundError(f"Provide the nine Olist CSV files before running: {missing}")

integer_cols = {
    "order_item_id", "payment_sequential", "payment_installments", "review_score",
    "product_name_lenght", "product_description_lenght", "product_photos_qty",
    "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
}
double_cols = {"price", "freight_value", "payment_value", "geolocation_lat", "geolocation_lng"}
timestamp_cols = {
    "shipping_limit_date", "review_creation_date", "review_answer_timestamp",
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
}
key_map = {
    "olist_customers_dataset": ["customer_id"],
    "olist_order_items_dataset": ["order_id", "order_item_id"],
    "olist_order_payments_dataset": ["order_id", "payment_sequential"],
    "olist_order_reviews_dataset": ["review_id", "order_id"],
    "olist_orders_dataset": ["order_id"], "olist_products_dataset": ["product_id"],
    "olist_sellers_dataset": ["seller_id"],
    "product_category_name_translation": ["product_category_name"],
}

def storage_mib(path):
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) / 1024**2

def null_profile(df):
    expressions = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns]
    return {k: int(v or 0) for k, v in df.agg(*expressions).first().asDict().items()}

parquet_dir.mkdir(parents=True, exist_ok=True)
summary = []
profiles = {}
# Read and write on local disk first, avoiding partial Spark writes on Drive/FUSE.
with tempfile.TemporaryDirectory(prefix="olist_ingestion_") as work:
    staging = Path(work)
    for filename in required_csvs:
        started = time.perf_counter()
        name = Path(filename).stem
        source = raw_dir / filename
        local_csv = staging / filename
        shutil.copy2(source, local_csv)
        # csv.reader counts logical records, including quoted multiline comments.
        with local_csv.open(encoding="utf-8-sig", newline="") as handle:
            reader = csv.reader(handle, strict=True)
            header = next(reader)
            expected_rows = 0
            for record in reader:
                if len(record) != len(header):
                    raise ValueError(f"{filename}: invalid field count at logical record {expected_rows + 1}")
                expected_rows += 1
        schema = StructType([StructField(c, StringType(), True) for c in header])
        df = (spark.read.schema(schema).option("header", True)
              .option("enforceSchema", False).option("multiLine", True)
              .option("quote", '"').option("escape", '"')
              .option("mode", "FAILFAST").csv(str(local_csv))).cache()
        if df.count() != expected_rows:
            raise ValueError(f"{filename}: Spark row count differs from CSV logical records")
        for c in header:
            target_type = "int" if c in integer_cols else "double" if c in double_cols else "timestamp" if c in timestamp_cols else None
            if target_type:
                converted = F.expr(f"try_cast(`{c}` AS {target_type})")
                if df.filter(F.col(c).isNotNull() & converted.isNull()).limit(1).count():
                    raise ValueError(f"{filename}: invalid {target_type} value in {c}")
                df = df.withColumn(c, converted)
            elif c.endswith("zip_code_prefix"):
                if df.filter(F.col(c).isNull() | ~F.col(c).rlike("^[0-9]{1,5}$")).limit(1).count():
                    raise ValueError(f"{filename}: invalid ZIP prefix")
                df = df.withColumn(c, F.lpad(c, 5, "0"))
            elif c.endswith("_id") and c != "order_item_id":
                if df.filter(F.col(c).isNull() | ~F.col(c).rlike("^[0-9a-f]{32}$")).limit(1).count():
                    raise ValueError(f"{filename}: invalid ID in {c}")
        if name in key_map:
            assert_unique(df, key_map[name], name)
        if "review_score" in header and df.filter(~F.col("review_score").between(1, 5)).limit(1).count():
            raise ValueError("Review score must be in 1..5")
        before_nulls = null_profile(df)
        local_parquet = staging / f"{name}.parquet"
        df.write.mode("overwrite").option("compression", "snappy").parquet(str(local_parquet))
        restored = spark.read.parquet(str(local_parquet))
        if restored.count() != expected_rows or null_profile(restored) != before_nulls:
            raise ValueError(f"{name}: Parquet round-trip validation failed")
        # Replace only this known generated table, after validating the staged copy.
        destination = parquet_dir / f"{name}.parquet"
        if destination.resolve().parent != parquet_dir.resolve():
            raise ValueError("Output path escaped parquet_dir")
        if destination.is_dir():
            shutil.rmtree(destination)
        elif destination.exists():
            destination.unlink()
        shutil.copytree(local_parquet, destination)
        final_files = list(destination.glob("*.parquet"))
        if not final_files or any(p.stat().st_size == 0 for p in final_files):
            raise ValueError(f"{name}: missing or empty copied Parquet part")
        final_df = spark.read.parquet(str(destination))
        if final_df.count() != expected_rows or null_profile(final_df) != before_nulls:
            raise ValueError(f"{name}: copied Parquet validation failed")
        csv_mib = source.stat().st_size / 1024**2
        pq_mib = storage_mib(destination)
        profiles[name] = before_nulls
        summary.append({
            "Table Name": name, "Row Count": expected_rows, "Column Count": len(header),
            "CSV Size (MiB)": csv_mib, "Parquet Size (MiB)": pq_mib,
            "Storage Savings (%)": 100 * (1 - pq_mib / csv_mib),
            "Total Null Values": sum(before_nulls.values()),
            "Processing Time (s)": time.perf_counter() - started,
            "Round Trip Validated": True,
        })
        print(f"{name}: {expected_rows:,} records; {csv_mib:.3f} -> {pq_mib:.3f} MiB; validated")
        spark.catalog.clearCache()

summary_df = pd.DataFrame(summary)
summary_df.to_csv(output_dir / "ingestion_summary_report.csv", index=False)
(output_dir / "ingestion_null_profile.json").write_text(json.dumps(profiles, indent=2), encoding="utf-8")
total_csv = summary_df["CSV Size (MiB)"].sum()
total_parquet = summary_df["Parquet Size (MiB)"].sum()
print(f"All nine datasets: {total_csv:.3f} -> {total_parquet:.3f} MiB; savings {100 * (1 - total_parquet / total_csv):.2f}%")


olist_customers_dataset: 99,441 records; 8.615 -> 6.720 MiB; validated
olist_geolocation_dataset: 1,000,163 records; 58.435 -> 16.430 MiB; validated
olist_order_items_dataset: 112,650 records; 14.723 -> 6.502 MiB; validated
olist_order_payments_dataset: 103,886 records; 5.510 -> 3.681 MiB; validated
olist_order_reviews_dataset: 99,224 records; 13.782 -> 9.235 MiB; validated
olist_orders_dataset: 99,441 records; 16.837 -> 10.228 MiB; validated
olist_products_dataset: 32,951 records; 2.269 -> 1.311 MiB; validated
olist_sellers_dataset: 3,095 records; 0.167 -> 0.126 MiB; validated
product_category_name_translation: 71 records; 0.002 -> 0.003 MiB; validated
All nine datasets: 120.341 -> 54.236 MiB; savings 54.93%
